# 🧹 Denoising Autoencoder on MNIST

**Project:** Build a deep learning model that removes noise from images using a Convolutional Autoencoder.

### Architecture Overview
```
Noisy Image → [Encoder: Conv layers] → Latent Space → [Decoder: ConvTranspose layers] → Clean Image
```

### What We'll Cover
1. Load and preprocess MNIST
2. Add Gaussian noise to images
3. Build a Convolutional Autoencoder in PyTorch
4. Train with MSE loss
5. Visualize noisy vs. reconstructed results
6. Evaluate with PSNR & SSIM metrics

## 1. Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install torch torchvision matplotlib scikit-image tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2. Load MNIST Dataset

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
NOISE_FACTOR = 0.4   # Controls how much noise is added (0.0 = none, 1.0 = heavy)
LEARNING_RATE = 1e-3
EPOCHS = 20
LATENT_DIM = 128     # Bottleneck size

# Transforms: only normalize to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL image to tensor and scales to [0,1]
])

# Download & load MNIST
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Training samples : {len(train_dataset):,}')
print(f'Test samples     : {len(test_dataset):,}')
print(f'Batches per epoch: {len(train_loader)}')

## 3. Add Gaussian Noise

We corrupt clean images by adding Gaussian noise and then clipping values back to [0, 1].

In [ ]:
def add_noise(images: torch.Tensor, noise_factor: float = 0.4) -> torch.Tensor:
    """
    Add Gaussian noise to a batch of images.
    
    Args:
        images      : Clean image tensor of shape (B, C, H, W), values in [0, 1]
        noise_factor: Standard deviation scale of the Gaussian noise
    Returns:
        Noisy image tensor clipped to [0, 1]
    """
    noise = torch.randn_like(images) * noise_factor
    noisy_images = images + noise
    return torch.clamp(noisy_images, 0.0, 1.0)


# ── Visualise clean vs. noisy ──────────────────────────────────────────────
sample_images, _ = next(iter(test_loader))
noisy_samples = add_noise(sample_images, NOISE_FACTOR)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
fig.suptitle('Top: Clean  |  Bottom: Noisy', fontsize=13, fontweight='bold')

for i in range(10):
    axes[0, i].imshow(sample_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(noisy_samples[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('clean_vs_noisy.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Noise factor: {NOISE_FACTOR}')

## 4. Build the Convolutional Autoencoder

```
ENCODER                          DECODER
Input (1×28×28)                  Latent (128)
  → Conv 32, 3×3, ReLU             → FC → reshape (32×7×7)
  → Conv 64, 3×3, ReLU, MaxPool2   → ConvT 64, 2×2, ReLU  (14×14)
  → Conv 64, 3×3, ReLU, MaxPool2   → ConvT 32, 2×2, ReLU  (28×28)
  → Flatten → FC(128)               → Conv 1, 3×3, Sigmoid
```

In [ ]:
class Encoder(nn.Module):
    """Convolutional Encoder: image → latent vector"""

    def __init__(self, latent_dim: int = 128):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Block 1 – 1×28×28 → 32×28×28
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Block 2 – 32×28×28 → 64×14×14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),     # ↓ spatial: 28→14

            # Block 3 – 64×14×14 → 64×7×7
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),     # ↓ spatial: 14→7
        )
        # Flatten 64×7×7 = 3136 → latent_dim
        self.fc = nn.Linear(64 * 7 * 7, latent_dim)

    def forward(self, x):
        x = self.conv_layers(x)          # (B, 64, 7, 7)
        x = x.view(x.size(0), -1)        # (B, 3136)
        return self.fc(x)                 # (B, latent_dim)


class Decoder(nn.Module):
    """Convolutional Decoder: latent vector → reconstructed image"""

    def __init__(self, latent_dim: int = 128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.deconv_layers = nn.Sequential(
            # Upsample 64×7×7 → 64×14×14
            nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Upsample 64×14×14 → 32×28×28
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Output conv: 32×28×28 → 1×28×28
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),   # Pixel values in [0, 1]
        )

    def forward(self, z):
        x = self.fc(z)                         # (B, 3136)
        x = x.view(x.size(0), 64, 7, 7)        # (B, 64, 7, 7)
        return self.deconv_layers(x)            # (B, 1, 28, 28)


class DenoisingAutoencoder(nn.Module):
    """Full Denoising Autoencoder = Encoder + Decoder"""

    def __init__(self, latent_dim: int = 128):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z = self.encoder(x)     # Compress to latent space
        return self.decoder(z)  # Reconstruct clean image


# Instantiate model
model = DenoisingAutoencoder(latent_dim=LATENT_DIM).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable:,}')
print()
print(model)

## 5. Training Setup

In [ ]:
criterion = nn.MSELoss()                         # Pixel-wise Mean Squared Error
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print('Loss     :', criterion)
print('Optimizer:', optimizer)
print('Scheduler:', scheduler)

## 6. Train the Model

In [ ]:
train_losses = []
val_losses   = []

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):

    # ── Training ────────────────────────────────────────────
    model.train()
    running_loss = 0.0

    for clean_imgs, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        clean_imgs = clean_imgs.to(device)
        noisy_imgs = add_noise(clean_imgs, NOISE_FACTOR).to(device)

        # Forward pass: reconstruct clean image from noisy input
        reconstructed = model(noisy_imgs)
        loss = criterion(reconstructed, clean_imgs)   # Compare with clean target

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * clean_imgs.size(0)

    train_loss = running_loss / len(train_dataset)
    train_losses.append(train_loss)

    # ── Validation ──────────────────────────────────────────
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for clean_imgs, _ in test_loader:
            clean_imgs = clean_imgs.to(device)
            noisy_imgs = add_noise(clean_imgs, NOISE_FACTOR).to(device)
            reconstructed = model(noisy_imgs)
            loss = criterion(reconstructed, clean_imgs)
            val_running_loss += loss.item() * clean_imgs.size(0)

    val_loss = val_running_loss / len(test_dataset)
    val_losses.append(val_loss)

    scheduler.step(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_denoising_autoencoder.pth')

    print(f'Epoch [{epoch:02d}/{EPOCHS}]  '
          f'Train Loss: {train_loss:.6f}  |  '
          f'Val Loss: {val_loss:.6f}  '
          f'{"✓ Best" if val_loss == best_val_loss else ""}')

print(f'\nBest Validation Loss: {best_val_loss:.6f}')

## 7. Plot Training & Validation Loss

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Train Loss', color='steelblue',  linewidth=2)
plt.plot(range(1, EPOCHS + 1), val_losses,   label='Val Loss',   color='darkorange', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Denoising Autoencoder – Training Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Visualise Results: Noisy → Reconstructed → Clean

In [ ]:
# Load the best saved model
model.load_state_dict(torch.load('best_denoising_autoencoder.pth', map_location=device))
model.eval()

# Get a batch of test images
test_imgs, _ = next(iter(test_loader))
test_imgs    = test_imgs.to(device)
noisy_imgs   = add_noise(test_imgs, NOISE_FACTOR).to(device)

with torch.no_grad():
    reconstructed = model(noisy_imgs)

# Move to CPU for plotting
test_imgs_cpu  = test_imgs.cpu().numpy()
noisy_imgs_cpu = noisy_imgs.cpu().numpy()
recon_cpu      = reconstructed.cpu().numpy()

n = 10   # Number of images to display
fig, axes = plt.subplots(3, n, figsize=(16, 5))
row_labels = ['Noisy Input', 'Reconstructed', 'Original Clean']

for i in range(n):
    for row, imgs in enumerate([noisy_imgs_cpu, recon_cpu, test_imgs_cpu]):
        axes[row, i].imshow(imgs[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[row, i].axis('off')
        if i == 0:
            axes[row, i].set_title(row_labels[row], fontsize=9, loc='left', pad=3)

plt.suptitle('Denoising Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('denoising_results.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Evaluate with PSNR & SSIM

| Metric | Meaning | Higher is better? |
|--------|---------|------------------|
| **PSNR** | Peak Signal-to-Noise Ratio (dB) | ✅ Yes |
| **SSIM** | Structural Similarity Index (0–1) | ✅ Yes |

In [ ]:
def evaluate_metrics(model, data_loader, noise_factor, device):
    """Compute average PSNR and SSIM on the full test set."""
    model.eval()
    psnr_noisy_list, psnr_recon_list = [], []
    ssim_noisy_list, ssim_recon_list = [], []

    with torch.no_grad():
        for clean_imgs, _ in tqdm(data_loader, desc='Evaluating'):
            clean_imgs = clean_imgs.to(device)
            noisy_imgs = add_noise(clean_imgs, noise_factor).to(device)
            recon_imgs = model(noisy_imgs)

            # Move to numpy (B, H, W)
            clean = clean_imgs.cpu().numpy().squeeze(1)
            noisy = noisy_imgs.cpu().numpy().squeeze(1)
            recon = recon_imgs.cpu().numpy().squeeze(1)

            for c, n, r in zip(clean, noisy, recon):
                psnr_noisy_list.append(psnr(c, n, data_range=1.0))
                psnr_recon_list.append(psnr(c, r, data_range=1.0))
                ssim_noisy_list.append(ssim(c, n, data_range=1.0))
                ssim_recon_list.append(ssim(c, r, data_range=1.0))

    return {
        'PSNR Noisy → Clean' : np.mean(psnr_noisy_list),
        'PSNR Recon → Clean' : np.mean(psnr_recon_list),
        'SSIM Noisy → Clean' : np.mean(ssim_noisy_list),
        'SSIM Recon → Clean' : np.mean(ssim_recon_list),
    }


metrics = evaluate_metrics(model, test_loader, NOISE_FACTOR, device)

print('\n' + '='*50)
print('         EVALUATION METRICS (Test Set)')
print('='*50)
for k, v in metrics.items():
    print(f'  {k:<26}: {v:.4f}')
print('='*50)

psnr_improvement = metrics['PSNR Recon → Clean'] - metrics['PSNR Noisy → Clean']
ssim_improvement = metrics['SSIM Recon → Clean'] - metrics['SSIM Noisy → Clean']
print(f'\n  PSNR Improvement : +{psnr_improvement:.4f} dB')
print(f'  SSIM Improvement : +{ssim_improvement:.4f}')

## 10. Effect of Different Noise Levels

In [ ]:
noise_levels = [0.1, 0.2, 0.4, 0.6, 0.8]
sample_img, _ = test_dataset[0]
sample_img = sample_img.unsqueeze(0).to(device)   # Add batch dim

fig, axes = plt.subplots(3, len(noise_levels), figsize=(15, 6))
fig.suptitle('Effect of Different Noise Levels', fontsize=13, fontweight='bold')

with torch.no_grad():
    for col, nf in enumerate(noise_levels):
        noisy = add_noise(sample_img, nf).to(device)
        recon = model(noisy)

        axes[0, col].imshow(sample_img.cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[0, col].set_title(f'Clean', fontsize=9)
        axes[0, col].axis('off')

        axes[1, col].imshow(noisy.cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[1, col].set_title(f'Noisy\n(σ={nf})', fontsize=9)
        axes[1, col].axis('off')

        axes[2, col].imshow(recon.cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[2, col].set_title(f'Reconstructed', fontsize=9)
        axes[2, col].axis('off')

plt.tight_layout()
plt.savefig('noise_level_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Latent Space Visualisation (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# Collect latent vectors for 2000 test samples
model.eval()
latents, labels_list = [], []

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        noisy = add_noise(imgs, NOISE_FACTOR).to(device)
        z = model.encoder(noisy)
        latents.append(z.cpu().numpy())
        labels_list.append(lbls.numpy())
        if len(latents) * BATCH_SIZE >= 2000:
            break

latents = np.vstack(latents)[:2000]
labels  = np.concatenate(labels_list)[:2000]

# t-SNE
print('Running t-SNE...')
tsne   = TSNE(n_components=2, random_state=42, perplexity=30)
latents_2d = tsne.fit_transform(latents)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1],
                      c=labels, cmap='tab10', alpha=0.6, s=8)
plt.colorbar(scatter, ticks=range(10), label='Digit Class')
plt.title('t-SNE of Latent Space (coloured by digit)', fontsize=13)
plt.xlabel('t-SNE Dim 1')
plt.ylabel('t-SNE Dim 2')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('tsne_latent_space.png', dpi=120, bbox_inches='tight')
plt.show()
print('Well-separated clusters → encoder has learned meaningful representations!')

## 12. Save & Load Model

In [ ]:
# Save full model (architecture + weights)
torch.save({
    'epoch'           : EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_val_loss'   : best_val_loss,
    'latent_dim'      : LATENT_DIM,
    'noise_factor'    : NOISE_FACTOR,
}, 'denoising_autoencoder_checkpoint.pth')

print('Model saved to denoising_autoencoder_checkpoint.pth')

# ── How to reload ────────────────────────────────────────────
# checkpoint = torch.load('denoising_autoencoder_checkpoint.pth')
# model = DenoisingAutoencoder(latent_dim=checkpoint['latent_dim']).to(device)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

## Summary

| Component | Detail |
|-----------|--------|
| **Model**      | Convolutional Autoencoder (Encoder + Decoder) |
| **Dataset**    | MNIST (60k train / 10k test) |
| **Noise Type** | Gaussian (σ = 0.4), clipped to [0,1] |
| **Loss**       | MSE between reconstruction and clean image |
| **Optimizer**  | Adam (lr=1e-3) + ReduceLROnPlateau |
| **Latent Dim** | 128 |
| **Metrics**    | PSNR, SSIM |

### Key Takeaways
- The autoencoder compresses noisy images into a compact latent representation and learns to filter out noise during reconstruction.
- PSNR and SSIM both improve significantly after denoising compared to the raw noisy images.
- The t-SNE plot shows the latent space organises digit classes into well-separated clusters — demonstrating meaningful feature learning.

### Possible Extensions
- Try **salt-and-pepper noise** or **Poisson noise** instead of Gaussian
- Use a **U-Net style skip connections** for better high-frequency detail recovery
- Train a **Variational Autoencoder (VAE)** for probabilistic denoising
- Apply to **colour images** (CIFAR-10) with 3-channel inputs